# Chapter 1: Probabilistic Modeling**Version**: 2.0**Coverage**: Sections 1.1-1.7---## Core Concept> A probabilistic model is a distribution over data whose shape is determined by model parameters. Our goal is to estimate or infer those parameters from observed data.

In [ ]:
# Import necessary librariesimport numpy as npimport matplotlib.pyplot as plt# from scipy.stats - using torch.distributions instead import poisson, gamma, binomfrom scipy.special import gamma as gamma_funcimport seaborn as sns# Set random seed for reproducibilitynp.random.seed(42)# Set plotting stylesns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (12, 5)plt.rcParams['font.size'] = 11print("All libraries imported successfully")

---## 1.1 Poisson Distribution and Maximum Likelihood Estimation

### The Poisson DistributionFor count data (spike counts):$$P(X = k | \lambda) = rac{\lambda^k e^{-\lambda}}{k!}$$Key property: E[X] = Var(X) = λ

In [ ]:
# ============================================# 1.1.1 Properties of Poisson Distribution# ============================================# Set true parameterlambda_true = 3.5  # Average 3.5 spikes per second# Generate simulated dataT = 100  # Observe 100 secondsspike_counts = np.random.poisson(lam=lambda_true, size=T)print("="*60)print("1.1 Simulated Poisson Data")print("="*60)print(f"\nTrue parameter λ = {lambda_true}")print(f"\nObserved data for first 20 seconds:")print(spike_counts[:20])print(f"\nData summary statistics:")print(f"  Sample mean:     {np.mean(spike_counts):.4f}")print(f"  Sample variance: {np.var(spike_counts, ddof=0):.4f}")print(f"  Theoretical mean:     {lambda_true}")print(f"  Theoretical variance: {lambda_true}")print(f"\n✓ 观察到的均值和方差都接近 λ = {lambda_true}")

In [ ]:
# ============================================# 1.1.2 Visualizing Poisson Distribution# ============================================fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Subplot 1: Theoretical distributionax = axes[0]k_values = np.arange(0, 12)pmf_values = poisson.pmf(k_values, mu=lambda_true)ax.bar(k_values, pmf_values, alpha=0.7, color='steelblue',        label=f'Poisson(λ={lambda_true})', edgecolor='black')ax.set_xlabel('Spike Count k', fontsize=12)ax.set_ylabel('Probability P(X=k)', fontsize=12)ax.set_title('Poisson PMF', fontsize=13, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)# Subplot 2: Observed histogram vs theoreticalax = axes[1]bins = np.arange(-0.5, 12.5, 1)ax.hist(spike_counts, bins=bins, alpha=0.6, density=True,         color='coral', edgecolor='black', label='Observed Histogram')# Overlay theoretical distributionax.plot(k_values, pmf_values, 'o-', linewidth=2.5, markersize=8,        color='darkblue', label='Theoretical')ax.set_xlabel('Spike Count k', fontsize=12)ax.set_ylabel('Frequency/Probability', fontsize=12)ax.set_title('Observed vs Theoretical', fontsize=13, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Poisson visualization complete")

### PropertiesMean-Variance Relationship:- For Poisson: mean = variance = λ- Test for Poisson: compare sample mean vs variance

---## Maximum Likelihood Estimation### The Likelihood Function

In [ ]:
# ============================================# 1.2.1 Computing MLE# ============================================def poisson_mle(observations):    """Compute MLE for Poisson parameter"""    return np.mean(observations)def poisson_log_likelihood(lambda_param, observations):    """计算泊松分布的对数似然"""    return np.sum(observations * np.log(lambda_param) - lambda_param)# 计算 MLElambda_mle = poisson_mle(spike_counts)print("="*60)print("1.2 最大似然估计 (MLE)")print("="*60)print(f"\n观察data: {len(spike_counts)} 秒的尖峰计数")print(f"data总和: {np.sum(spike_counts)} 个尖峰")print(f"\nMLE 估计: λ̂ = {lambda_mle:.4f}")print(f"真实parameter: λ  = {lambda_true:.4f}")print(f"估计误差: {abs(lambda_mle - lambda_true):.4f}")print(f"\n✓ MLE 是样本平均值 = {np.mean(spike_counts):.4f}")

In [ ]:
# ============================================# 1.2.2 可视化对数似然函数# ============================================fig, axes = plt.subplots(1, 2, figsize=(14, 5))# 创建parameter范围lambda_range = np.linspace(0.5, 6, 200)# 计算对数似然log_likes = np.array([poisson_log_likelihood(lam, spike_counts)                       for lam in lambda_range])# 子图1：对数似然函数ax = axes[0]ax.plot(lambda_range, log_likes, 'b-', linewidth=2.5, label='对数似然函数')ax.axvline(lambda_mle, color='red', linestyle='--', linewidth=2.5,            label=f'MLE: λ̂ = {lambda_mle:.3f}')ax.axvline(lambda_true, color='green', linestyle='--', linewidth=2.5,           label=f'真实: λ = {lambda_true:.3f}')# 标注最大值max_idx = np.argmax(log_likes)ax.plot(lambda_range[max_idx], log_likes[max_idx], 'r*', markersize=20)ax.set_xlabel('parameter λ', fontsize=12)ax.set_ylabel('对数似然 log L(λ)', fontsize=12)ax.set_title('泊松分布的对数似然函数', fontsize=13, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)# 子图2：似然函数（归一化）ax = axes[1]likes = np.exp(log_likes - np.max(log_likes))  # 归一化避免溢出ax.plot(lambda_range, likes, 'b-', linewidth=2.5, label='似然函数 (归一化)')ax.axvline(lambda_mle, color='red', linestyle='--', linewidth=2.5,           label=f'MLE: λ̂ = {lambda_mle:.3f}')ax.fill_between(lambda_range, likes, alpha=0.3, color='steelblue')ax.set_xlabel('parameter λ', fontsize=12)ax.set_ylabel('似然 (归一化)', fontsize=12)ax.set_title('似然函数的几何意义', fontsize=13, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("✓ MLE 图表已生成")

Given observations X₁, ..., Xₜ:$$\hat{\lambda}_{MLE} = rac{1}{T}\sum_t X_t$$The MLE is simply the sample mean!

### Visualizing the Likelihood

In [ ]:
# ============================================# 1.3.1 不同的先验分布# ============================================from scipy.stats import gamma as gamma_dist# 不同的先验配置prior_configs = [    (1, 1, '无信息先验'),    (2, 1, '弱先验'),    (5, 2, '中等先验'),    (10, 3, '强先验 (倾向小λ)'),]print("="*60)print("1.3 贝叶斯先验：伽马分布")print("="*60)print("\n先验配置比较:")print(f"{'先验名称':<20} {'α':<5} {'β':<5} {'E[λ]':<8} {'Var(λ)'}")print("-" * 70)for alpha, beta, name in prior_configs:    mean = alpha / beta    var = alpha / (beta ** 2)    print(f"{name:<20} {alpha:<5} {beta:<5} {mean:<8.2f} {var:.2f}")

In [ ]:
# ============================================# 1.3.2 可视化不同的先验# ============================================fig, axes = plt.subplots(2, 2, figsize=(14, 10))lambda_range = np.linspace(0, 12, 1000)for idx, (alpha, beta, name) in enumerate(prior_configs):    ax = axes[idx // 2, idx % 2]        # 计算PDF    pdf = gamma_dist.pdf(lambda_range, a=alpha, scale=1/beta)        # 绘制    ax.plot(lambda_range, pdf, 'b-', linewidth=2.5)    ax.fill_between(lambda_range, pdf, alpha=0.3, color='steelblue')        # 标注期望值    mean = alpha / beta    ax.axvline(mean, color='red', linestyle='--', linewidth=2,               label=f'E[λ] = {mean:.2f}')        # 标注真实值（供参考）    ax.axvline(lambda_true, color='green', linestyle=':', linewidth=2,               label=f'真实 λ = {lambda_true:.2f}')        ax.set_xlabel('λ', fontsize=11)    ax.set_ylabel('概率密度', fontsize=11)    ax.set_title(f'{name}: Ga(α={alpha}, β={beta})',                 fontsize=12, fontweight='bold')    ax.legend(fontsize=10)    ax.grid(True, alpha=0.3)    ax.set_xlim(0, 12)plt.tight_layout()plt.show()print("\n✓ 先验分布可视化完成")

---## 1.3 Bayesian Inference### The Bayesian Framework

Posterior distribution:$$P(\lambda | X) = rac{P(X|\lambda) P(\lambda)}{P(X)}$$Full uncertainty quantification through distribution.

In [ ]:
# ============================================# 1.4.1 完整的贝叶斯推断# ============================================# 设定先验alpha_prior = 2beta_prior = 1prior_mean = alpha_prior / beta_prior# 计算后验parametern_obs = len(spike_counts)total_spikes = np.sum(spike_counts)alpha_post = alpha_prior + total_spikesbeta_post = beta_prior + n_obspost_mean = alpha_post / beta_postprint("="*60)print("1.4 后验分布与贝叶斯推断")print("="*60)print(f"\n先验: Ga(α={alpha_prior}, β={beta_prior})")print(f"  先验均值: E[λ] = {prior_mean:.3f}")print(f"\ndata:")print(f"  观察数: {n_obs}")print(f"  总尖峰数: {total_spikes}")print(f"  样本均值 (MLE): {np.mean(spike_counts):.3f}")print(f"\n后验: Ga(α={alpha_post}, β={beta_post})")print(f"  后验均值: E[λ|D] = {post_mean:.3f}")# 计算可信区间（Credible Interval）credible_lower = gamma_dist.ppf(0.025, a=alpha_post, scale=1/beta_post)credible_upper = gamma_dist.ppf(0.975, a=alpha_post, scale=1/beta_post)print(f"\n95% 可信区间 (Credible Interval):")print(f"  P(λ ∈ [{credible_lower:.3f}, {credible_upper:.3f}] | D) = 0.95")print(f"  真实值 λ = {lambda_true} 在区间内? ", end="")print("✓ 是" if credible_lower <= lambda_true <= credible_upper else "✗ 否")

In [ ]:
# ============================================# 1.4.2 完整的贝叶斯推断可视化# ============================================fig, axes = plt.subplots(2, 2, figsize=(14, 11))lambda_range = np.linspace(0, 8, 1000)# ==================# 子图1：先验分布# ==================ax = axes[0, 0]prior_pdf = gamma_dist.pdf(lambda_range, a=alpha_prior, scale=1/beta_prior)ax.plot(lambda_range, prior_pdf, 'b-', linewidth=2.5, label='先验分布')ax.fill_between(lambda_range, prior_pdf, alpha=0.3, color='steelblue')ax.axvline(prior_mean, color='blue', linestyle='--', linewidth=2,           label=f'先验均值 = {prior_mean:.3f}')ax.set_xlabel('λ', fontsize=11)ax.set_ylabel('密度', fontsize=11)ax.set_title(f'(a) 先验: Ga({alpha_prior}, {beta_prior})', fontsize=12, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)# ==================# 子图2：似然函数# ==================ax = axes[0, 1]lambda_range_fine = np.linspace(2, 5, 1000)likelihood = np.exp(total_spikes * np.log(lambda_range_fine) - n_obs * lambda_range_fine)likelihood = likelihood / np.max(likelihood)  # 归一化ax.plot(lambda_range_fine, likelihood, 'g-', linewidth=2.5, label='似然函数 (归一化)')ax.fill_between(lambda_range_fine, likelihood, alpha=0.3, color='green')ax.axvline(np.mean(spike_counts), color='green', linestyle='--', linewidth=2,           label=f'MLE = {np.mean(spike_counts):.3f}')ax.set_xlabel('λ', fontsize=11)ax.set_ylabel('似然 (归一化)', fontsize=11)ax.set_title(f'(b) 似然: {n_obs} 个观察，{total_spikes} 个尖峰',              fontsize=12, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)# ==================# 子图3：先验、似然、后验叠加# ==================ax = axes[1, 0]lambda_range = np.linspace(0, 8, 1000)prior_pdf = gamma_dist.pdf(lambda_range, a=alpha_prior, scale=1/beta_prior)post_pdf = gamma_dist.pdf(lambda_range, a=alpha_post, scale=1/beta_post)# 似然需要重新计算并缩放likelihood_full = np.exp(total_spikes * np.log(lambda_range + 0.01)                          - n_obs * lambda_range)likelihood_full = likelihood_full / np.max(likelihood_full) * 0.6  # 缩放用于可视化ax.plot(lambda_range, prior_pdf, 'b-', linewidth=2.5, label='先验')ax.plot(lambda_range, likelihood_full, 'g-', linewidth=2.5, label='似然 (缩放)')ax.plot(lambda_range, post_pdf, 'r-', linewidth=2.5, label='后验')ax.fill_between(lambda_range, post_pdf, alpha=0.2, color='red')ax.axvline(lambda_true, color='purple', linestyle=':', linewidth=2.5,           label=f'真实 λ = {lambda_true}')ax.set_xlabel('λ', fontsize=11)ax.set_ylabel('密度', fontsize=11)ax.set_title('(c) 从先验到后验的更新', fontsize=12, fontweight='bold')ax.legend(fontsize=10, loc='upper right')ax.grid(True, alpha=0.3)# ==================# 子图4：后验分布 + 可信区间# ==================ax = axes[1, 1]lambda_range = np.linspace(0, 8, 1000)post_pdf = gamma_dist.pdf(lambda_range, a=alpha_post, scale=1/beta_post)ax.plot(lambda_range, post_pdf, 'r-', linewidth=2.5)ax.fill_between(lambda_range, post_pdf, alpha=0.3, color='salmon')# 绘制可信区间mask = (lambda_range >= credible_lower) & (lambda_range <= credible_upper)ax.fill_between(lambda_range[mask], post_pdf[mask], alpha=0.6,                 color='darkred', label='95% 可信区间')ax.axvline(post_mean, color='darkred', linestyle='-', linewidth=2.5,           label=f'后验均值 = {post_mean:.3f}')ax.axvline(credible_lower, color='gray', linestyle='--', linewidth=1.5)ax.axvline(credible_upper, color='gray', linestyle='--', linewidth=1.5)ax.axvline(lambda_true, color='purple', linestyle=':', linewidth=2.5,           label=f'真实 λ = {lambda_true}')ax.set_xlabel('λ', fontsize=11)ax.set_ylabel('密度', fontsize=11)ax.set_title(f'(d) 后验: Ga({alpha_post}, {beta_post})',              fontsize=12, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("✓ 贝叶斯推断完整可视化生成")

In [ ]:
# ============================================# 1.4.3 频率派 vs 贝叶斯派比较# ============================================mle_est = np.mean(spike_counts)bayes_est = post_meanprint("="*60)print("频率派 (Frequentist) vs 贝叶斯派 (Bayesian)")print("="*60)print(f"\n📊 频率派 (MLE):")print(f"  点估计: λ̂ = {mle_est:.4f}")print(f"  解释: '这是对parameter的最佳猜测'")print(f"  不确定性: 需要额外计算（如bootstrap或渐近分布）")print(f"\n🎲 贝叶斯派:")print(f"  点估计 (后验均值): λ = {bayes_est:.4f}")print(f"  完整分布: Ga({alpha_post}, {beta_post})")print(f"  可信区间: 95% CI = [{credible_lower:.4f}, {credible_upper:.4f}]")print(f"  解释: '给定data，λ的后验分布是...'")print(f"\n⚖️  关键区别:")print(f"  MLE ({mle_est:.4f}) vs 贝叶斯 ({bayes_est:.4f})")print(f"  差异: {abs(mle_est - bayes_est):.4f}")print(f"\n  贝叶斯估计介于先验均值 ({prior_mean:.4f}) 和 MLE 之间")print(f"  这就是'正则化'：向先验知识靠拢")

### Conjugate PriorsFor Poisson, use Gamma prior:$$P(\lambda|X) = 	ext{Gamma}(lpha + \sum X_t, eta + T)$$Closed-form posterior!

---## 1.4 Mixture Models### Motivation for Mixture Models

Mixture model:$$P(X) = \sum_{k=1}^K \pi_k P(X | 	heta_k)$$Multiple components with mixing weights.

In [ ]:
# ============================================# 1.4.4 比较 MLE、MAP 和后验均值# ============================================# 计算 MAP 估计（伽马分布的众数）map_estimate = max(0.01, (alpha_post - 1) / beta_post)  # 避免负值print("="*60)print("1.4.4 MLE vs MAP vs 后验均值")print("="*60)print(f"\n📊 频率派 (MLE):")print(f"  λ̂_MLE = {mle_est:.4f}")print(f"  最大化: L(λ|D) = P(D|λ)")print(f"  含义: '最能解释data的单个parameter值'")print(f"\n🎲 贝叶斯派 - MAP 估计:")print(f"  λ̂_MAP = {map_estimate:.4f}")print(f"  最大化: P(λ|D) ∝ P(D|λ)·P(λ)")print(f"  含义: '后验分布最可能的值'")print(f"\n🎲 贝叶斯派 - 后验均值:")print(f"  E[λ|D] = {post_mean:.4f}")print(f"  计算: ∫ λ·P(λ|D) dλ")print(f"  含义: '后验分布的平均值'")print(f"\n📈 数值比较:")print(f"  {'估计方法':<15} {'值':<10} {'误差':<10}")print(f"  {'-'*35}")print(f"  {'真实值':<15} {lambda_true:<10.4f} {'基准':<10}")print(f"  {'MLE':<15} {mle_est:<10.4f} {abs(mle_est-lambda_true):<10.4f}")print(f"  {'MAP':<15} {map_estimate:<10.4f} {abs(map_estimate-lambda_true):<10.4f}")print(f"  {'后验均值':<15} {post_mean:<10.4f} {abs(post_mean-lambda_true):<10.4f}")# 可视化对比fig, ax = plt.subplots(figsize=(12, 6))lambda_range = np.linspace(1, 6, 1000)post_pdf = gamma_dist.pdf(lambda_range, a=alpha_post, scale=1/beta_post)ax.plot(lambda_range, post_pdf, 'b-', linewidth=3, label='后验分布')ax.fill_between(lambda_range, post_pdf, alpha=0.2, color='steelblue')# 标注三种估计ax.axvline(mle_est, color='red', linestyle='-', linewidth=2.5, label=f'MLE = {mle_est:.3f}')ax.axvline(map_estimate, color='orange', linestyle='--', linewidth=2.5, label=f'MAP = {map_estimate:.3f}')ax.axvline(post_mean, color='green', linestyle='-.', linewidth=2.5, label=f'E[λ|D] = {post_mean:.3f}')ax.axvline(lambda_true, color='purple', linestyle=':', linewidth=2.5, label=f'真实 λ = {lambda_true:.3f}')ax.set_xlabel('λ', fontsize=12, fontweight='bold')ax.set_ylabel('后验概率密度', fontsize=12, fontweight='bold')ax.set_title('后验分布中的三种点估计', fontsize=13, fontweight='bold')ax.legend(fontsize=11, loc='upper right')ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n✓ MLE vs MAP vs 后验均值对比完成")

### Gaussian Mixture Model$$P(X) = \sum_{k=1}^K \pi_k \mathcal{N}(X|\mu_k, \sigma_k^2)$$Challenge: Likelihood doesn't factor nicely.

In [ ]:
# ============================================# 1.5.1 生成泊松混合模型data# ============================================# 混合模型parameter（真实值）K_true = 2  # 两个神经元状态pi_true = np.array([0.4, 0.6])  # 混合权重lambda_true_mixture = np.array([2.0, 4.5])  # 两个状态的发火率print("="*60)print("1.5 泊松混合模型")print("="*60)print(f"\n真实混合模型parameter:")print(f"  K = {K_true} 个分量")print(f"  π = {pi_true}  (混合权重)")print(f"  λ = {lambda_true_mixture}  (发火率)")# 生成混合datanp.random.seed(42)T_mixture = 200  # 观察点数z_true = np.random.choice(K_true, size=T_mixture, p=pi_true)  # 隐变量spike_counts_mixture = np.array([np.random.poisson(lambda_true_mixture[z]) for z in z_true])print(f"\n生成data:")print(f"  样本数: {T_mixture}")print(f"  来自分量1 (λ={lambda_true_mixture[0]}) 的样本: {np.sum(z_true==0)}")print(f"  来自分量2 (λ={lambda_true_mixture[1]}) 的样本: {np.sum(z_true==1)}")print(f"\nData summary statistics:")print(f"  均值: {np.mean(spike_counts_mixture):.3f}")print(f"  方差: {np.var(spike_counts_mixture):.3f}")print(f"  预期混合均值: {np.sum(pi_true * lambda_true_mixture):.3f}")

In [ ]:
# ============================================# 1.5.2 可视化混合data# ============================================fig, axes = plt.subplots(2, 2, figsize=(14, 10))# 子图1：观察data直方图ax = axes[0, 0]ax.hist(spike_counts_mixture, bins=np.arange(0, 12) - 0.5, alpha=0.7,        color='steelblue', edgecolor='black', density=True)ax.set_xlabel('尖峰计数', fontsize=11)ax.set_ylabel('频率', fontsize=11)ax.set_title('(a) 观察data的边际分布', fontsize=12, fontweight='bold')ax.grid(True, alpha=0.3)# 子图2：真实分量的理论分布ax = axes[0, 1]k_vals = np.arange(0, 10)for k in range(K_true):    pmf = poisson.pmf(k_vals, lambda_true_mixture[k])    ax.bar(k_vals + k*0.35, pi_true[k] * pmf, width=0.35,           alpha=0.6, label=f'分量{k+1} (π={pi_true[k]:.1f}, λ={lambda_true_mixture[k]:.1f})')ax.set_xlabel('尖峰计数', fontsize=11)ax.set_ylabel('概率', fontsize=11)ax.set_title('(b) 真实混合分布的分量', fontsize=12, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)# 子图3：时间序列（按真实分量着色）ax = axes[1, 0]colors = ['red' if z==0 else 'blue' for z in z_true]ax.scatter(range(T_mixture), spike_counts_mixture, c=colors, alpha=0.6, s=30)ax.axhline(lambda_true_mixture[0], color='red', linestyle='--', alpha=0.5, label='分量1均值')ax.axhline(lambda_true_mixture[1], color='blue', linestyle='--', alpha=0.5, label='分量2均值')ax.set_xlabel('时间 t', fontsize=11)ax.set_ylabel('尖峰计数', fontsize=11)ax.set_title('(c) 时间序列（真实隐变量已知）', fontsize=12, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)# 子图4：箱线图对比两个分量ax = axes[1, 1]data_by_component = [spike_counts_mixture[z_true==k] for k in range(K_true)]bp = ax.boxplot(data_by_component, labels=[f'分量{k+1}' for k in range(K_true)],                patch_artist=True)colors_box = ['lightcoral', 'lightblue']for patch, color in zip(bp['boxes'], colors_box):    patch.set_facecolor(color)ax.set_ylabel('尖峰计数', fontsize=11)ax.set_title('(d) 两个分量的data分布', fontsize=12, fontweight='bold')ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()print("✓ 混合模型data可视化完成")

---## 1.5 The Expectation-Maximization (EM) Algorithm### The Missing Data Perspective

In [ ]:
# ============================================# 1.6.1 实现 EM 算法# ============================================def em_poisson_mixture(X, K, max_iter=100, tol=1e-4, verbose=False):    """    泊松混合模型的 EM 算法    parameter:        X: 观察data (N,)        K: 分量数        max_iter: 最大迭代次数        tol: 收敛容限        verbose: 是否打印迭代信息    返回:        pi: 混合权重 (K,)        lambda_: 发火率 (K,)        gamma: 隐变量后验 (N, K)        log_like: 对数似然轨迹    """    N = len(X)    # 初始化parameter    pi = np.ones(K) / K    lambda_ = np.linspace(np.min(X), np.max(X), K) + np.random.rand(K) * 0.5    lambda_ = np.maximum(lambda_, 0.1)  # 确保正数    gamma = np.zeros((N, K))  # 隐变量后验    log_like_trace = []    if verbose:        print(f"\n{'迭代':<8} {'对数似然':<15} {'变化':<15}")        print("-" * 40)    for iteration in range(max_iter):        # ========== E-Step ==========        # 计算 P(z_t=k | x_t)        for k in range(K):            # P(x_t | z_t=k) * P(z_t=k)            gamma[:, k] = poisson.pmf(X, lambda_[k]) * pi[k]        # 归一化为概率        gamma = gamma / (np.sum(gamma, axis=1, keepdims=True) + 1e-10)        # 计算对数似然        log_like = 0        for i in range(N):            marginal_prob = np.sum([pi[k] * poisson.pmf(X[i], lambda_[k]) for k in range(K)])            log_like += np.log(marginal_prob + 1e-10)        log_like_trace.append(log_like)        # ========== M-Step ==========        # 更新混合权重        N_k = np.sum(gamma, axis=0)  # 每个分量的有效样本数        pi = N_k / N        # 更新发火率        for k in range(K):            lambda_[k] = np.sum(gamma[:, k] * X) / (N_k[k] + 1e-10)        # 检查收敛        if iteration > 0:            change = abs(log_like_trace[-1] - log_like_trace[-2])            if verbose and iteration % 10 == 0:                print(f"{iteration:<8} {log_like_trace[-1]:<15.2f} {change:<15.6f}")            if change < tol:                if verbose:                    print(f"\n收敛于迭代 {iteration}")                break    return pi, lambda_, gamma, np.array(log_like_trace)# 运行 EM 算法print("="*60)print("1.6 期望最大化 (EM) 算法")print("="*60)print(f"\n初始条件:")print(f"  data: {T_mixture} 个观察")print(f"  真实parameter: π={pi_true}, λ={lambda_true_mixture}")print(f"  隐变量: 未知")# 设定要估计的分量数（假设我们知道 K=2）K_est = 2pi_em, lambda_em, gamma_em, log_like_em = em_poisson_mixture(    spike_counts_mixture, K_est, max_iter=100, verbose=True)print(f"\n\n📊 EM 算法估计结果:")print(f"  π_hat = {pi_em}")print(f"  λ_hat = {lambda_em}")print(f"\n📊 真实parameter:")print(f"  π_true = {pi_true}")print(f"  λ_true = {lambda_true_mixture}")print(f"\n📈 parameter估计误差:")# 需要处理分量顺序的可能不同dist1 = np.sum((pi_em - pi_true)**2) + np.sum((lambda_em - lambda_true_mixture)**2)dist2 = np.sum((pi_em[::-1] - pi_true)**2) + np.sum((lambda_em[::-1] - lambda_true_mixture)**2)if dist1 <= dist2:    print(f"  π 误差: {np.linalg.norm(pi_em - pi_true):.4f}")    print(f"  λ 误差: {np.linalg.norm(lambda_em - lambda_true_mixture):.4f}")else:    print(f"  注意：分量顺序可能反转")    print(f"  π 误差 (调整顺序): {np.linalg.norm(pi_em[::-1] - pi_true):.4f}")    print(f"  λ 误差 (调整顺序): {np.linalg.norm(lambda_em[::-1] - lambda_true_mixture):.4f}")

In [ ]:
# ============================================# 1.6.2 可视化 EM 算法收敛# ============================================fig, axes = plt.subplots(2, 2, figsize=(14, 10))# 子图1：对数似然轨迹ax = axes[0, 0]ax.plot(log_like_em, 'b-', linewidth=2.5, marker='o', markersize=4)ax.set_xlabel('EM 迭代次数', fontsize=11)ax.set_ylabel('对数似然', fontsize=11)ax.set_title('(a) EM 算法的对数似然收敛', fontsize=12, fontweight='bold')ax.grid(True, alpha=0.3)# 子图2：隐变量分配ax = axes[0, 1]im = ax.imshow(gamma_em[:100].T, aspect='auto', cmap='RdYlBu_r')ax.set_xlabel('样本索引', fontsize=11)ax.set_ylabel('分量', fontsize=11)ax.set_title('(b) 隐变量软分配 (γ_{t,k}) 前100个样本', fontsize=12, fontweight='bold')ax.set_yticks([0, 1])ax.set_yticklabels(['分量1', '分量2'])plt.colorbar(im, ax=ax, label='概率')# 子图3：估计 vs 真实parameterax = axes[1, 0]x_pos = np.arange(K_est)width = 0.35ax.bar(x_pos - width/2, pi_true, width, label='真实 π', alpha=0.7, color='steelblue')ax.bar(x_pos + width/2, pi_em, width, label='估计 π', alpha=0.7, color='coral')ax.set_xlabel('分量', fontsize=11)ax.set_ylabel('混合权重', fontsize=11)ax.set_title('(c) 混合权重估计对比', fontsize=12, fontweight='bold')ax.set_xticks(x_pos)ax.set_xticklabels([f'k={k+1}' for k in range(K_est)])ax.legend(fontsize=10)ax.grid(True, alpha=0.3, axis='y')# 子图4：发火率估计对比ax = axes[1, 1]ax.bar(x_pos - width/2, lambda_true_mixture, width, label='真实 λ', alpha=0.7, color='steelblue')ax.bar(x_pos + width/2, lambda_em, width, label='估计 λ', alpha=0.7, color='coral')ax.set_xlabel('分量', fontsize=11)ax.set_ylabel('发火率', fontsize=11)ax.set_title('(d) 发火率估计对比', fontsize=12, fontweight='bold')ax.set_xticks(x_pos)ax.set_xticklabels([f'k={k+1}' for k in range(K_est)])ax.legend(fontsize=10)ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()print("\n✓ EM 算法收敛过程可视化完成")

View mixture with latent variables Z_i ∈ {1,...,K}.If we knew Z, it's easy: fit each component separately!

In [ ]:
# ============================================# 1.7.1 模型选择：AIC 和 BIC# ============================================def compute_model_criteria(X, K, pi, lambda_):    """    计算 AIC 和 BIC    parameter:        X: 观察data        K: 分量数        pi: 混合权重        lambda_: 发火率    返回:        log_like: 对数似然        aic: AIC        bic: BIC    """    N = len(X)    # 计算对数似然    log_like = 0    for i in range(N):        marginal = np.sum([pi[k] * poisson.pmf(X[i], lambda_[k]) for k in range(K)])        log_like += np.log(marginal + 1e-10)    # parameter数量：K 个权重（但有一个约束 Σπ_k=1），K 个发火率    # 实际parameter数：(K-1) + K = 2K - 1    p = 2 * K - 1    # AIC = -2*LL + 2*p    aic = -2 * log_like + 2 * p    # BIC = -2*LL + p*log(N)    bic = -2 * log_like + p * np.log(N)    return log_like, aic, bic# 尝试不同的 K 值K_values = np.arange(1, 6)log_likes = []aics = []bics = []results = []print("="*60)print("1.7 模型选择")print("="*60)print(f"\n尝试不同分量数 K...\n")print(f"{'K':<5} {'对数似然':<15} {'AIC':<15} {'BIC':<15} {'parameter数':<8}")print("-" * 60)for K in K_values:    pi_k, lambda_k, gamma_k, _ = em_poisson_mixture(spike_counts_mixture, K, max_iter=100, verbose=False)    ll, aic, bic = compute_model_criteria(spike_counts_mixture, K, pi_k, lambda_k)    log_likes.append(ll)    aics.append(aic)    bics.append(bic)    results.append((K, pi_k, lambda_k))    p = 2 * K - 1    print(f"{K:<5} {ll:<15.2f} {aic:<15.2f} {bic:<15.2f} {p:<8}")# 找到最优模型best_k_aic = K_values[np.argmin(aics)]best_k_bic = K_values[np.argmin(bics)]print(f"\n📊 模型选择结果:")print(f"  AIC 最优: K = {best_k_aic}")print(f"  BIC 最优: K = {best_k_bic}")print(f"  真实值: K = {K_true}")if best_k_bic == K_true:    print(f"\n  ✓ BIC 正确识别了真实模型!")else:    print(f"\n  注意：BIC 选择的模型与真实值不同")    print(f"  （这可能是由于样本量、parameter值等因素）")

In [ ]:
# ============================================# 1.7.2 可视化模型选择# ============================================fig, axes = plt.subplots(1, 3, figsize=(15, 5))# 子图1：对数似然ax = axes[0]ax.plot(K_values, log_likes, 'o-', linewidth=2.5, markersize=10, color='steelblue')ax.set_xlabel('分量数 K', fontsize=12, fontweight='bold')ax.set_ylabel('对数似然', fontsize=12, fontweight='bold')ax.set_title('(a) 对数似然 vs K', fontsize=12, fontweight='bold')ax.set_xticks(K_values)ax.grid(True, alpha=0.3)# 子图2：AICax = axes[1]ax.plot(K_values, aics, 'o-', linewidth=2.5, markersize=10, color='coral')ax.axvline(best_k_aic, color='red', linestyle='--', linewidth=2, label=f'AIC 最优: K={best_k_aic}')ax.axvline(K_true, color='green', linestyle=':', linewidth=2, label=f'真实: K={K_true}')ax.set_xlabel('分量数 K', fontsize=12, fontweight='bold')ax.set_ylabel('AIC', fontsize=12, fontweight='bold')ax.set_title('(b) AIC vs K', fontsize=12, fontweight='bold')ax.set_xticks(K_values)ax.legend(fontsize=10)ax.grid(True, alpha=0.3)# 子图3：BICax = axes[2]ax.plot(K_values, bics, 'o-', linewidth=2.5, markersize=10, color='lightgreen')ax.axvline(best_k_bic, color='red', linestyle='--', linewidth=2, label=f'BIC 最优: K={best_k_bic}')ax.axvline(K_true, color='green', linestyle=':', linewidth=2, label=f'真实: K={K_true}')ax.set_xlabel('分量数 K', fontsize=12, fontweight='bold')ax.set_ylabel('BIC', fontsize=12, fontweight='bold')ax.set_title('(c) BIC vs K', fontsize=12, fontweight='bold')ax.set_xticks(K_values)ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n✓ 模型选择可视化完成")

### EM AlgorithmE-step: Compute responsibilities γᵢₖ = P(Zᵢ=k|Xᵢ,θ)M-step: Refit parameters using responsibilities as weightsEM increases likelihood each iteration.

---## 1.6 Model Selection### The Model Selection Problem

Choose complexity (K components, which features, etc).Problem: Complex models always fit better!Solution: Penalize complexity.

### Information CriteriaAIC = -2ℓ(θ̂) + 2pBIC = -2ℓ(θ̂) + p log(T)Lower is better. BIC has stronger penalty.

In [ ]:
# ============================================# 1.4.4 比较 MLE、MAP 和后验均值# ============================================# 计算 MAP 估计（伽马分布的众数）map_estimate = max(0.01, (alpha_post - 1) / beta_post)  # 避免负值print("="*60)print("1.4.4 MLE vs MAP vs 后验均值")print("="*60)print(f"\n📊 频率派 (MLE):")print(f"  λ̂_MLE = {mle_est:.4f}")print(f"  最大化: L(λ|D) = P(D|λ)")print(f"  含义: '最能解释data的单个parameter值'")print(f"\n🎲 贝叶斯派 - MAP 估计:")print(f"  λ̂_MAP = {map_estimate:.4f}")print(f"  最大化: P(λ|D) ∝ P(D|λ)·P(λ)")print(f"  含义: '后验分布最可能的值'")print(f"\n🎲 贝叶斯派 - 后验均值:")print(f"  E[λ|D] = {post_mean:.4f}")print(f"  计算: ∫ λ·P(λ|D) dλ")print(f"  含义: '后验分布的平均值'")print(f"\n📈 数值比较:")print(f"  {'估计方法':<15} {'值':<10} {'误差':<10}")print(f"  {'-'*35}")print(f"  {'真实值':<15} {lambda_true:<10.4f} {'基准':<10}")print(f"  {'MLE':<15} {mle_est:<10.4f} {abs(mle_est-lambda_true):<10.4f}")print(f"  {'MAP':<15} {map_estimate:<10.4f} {abs(map_estimate-lambda_true):<10.4f}")print(f"  {'后验均值':<15} {post_mean:<10.4f} {abs(post_mean-lambda_true):<10.4f}")# 可视化对比fig, ax = plt.subplots(figsize=(12, 6))lambda_range = np.linspace(1, 6, 1000)post_pdf = gamma_dist.pdf(lambda_range, a=alpha_post, scale=1/beta_post)ax.plot(lambda_range, post_pdf, 'b-', linewidth=3, label='后验分布')ax.fill_between(lambda_range, post_pdf, alpha=0.2, color='steelblue')# 标注三种估计ax.axvline(mle_est, color='red', linestyle='-', linewidth=2.5, label=f'MLE = {mle_est:.3f}')ax.axvline(map_estimate, color='orange', linestyle='--', linewidth=2.5, label=f'MAP = {map_estimate:.3f}')ax.axvline(post_mean, color='green', linestyle='-.', linewidth=2.5, label=f'E[λ|D] = {post_mean:.3f}')ax.axvline(lambda_true, color='purple', linestyle=':', linewidth=2.5, label=f'真实 λ = {lambda_true:.3f}')ax.set_xlabel('λ', fontsize=12, fontweight='bold')ax.set_ylabel('后验概率密度', fontsize=12, fontweight='bold')ax.set_title('后验分布中的三种点估计', fontsize=13, fontweight='bold')ax.legend(fontsize=11, loc='upper right')ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n✓ MLE vs MAP vs 后验均值对比完成")

---## 1.7 Maximum A Posteriori (MAP) Estimation### Combining Prior and Likelihood

In [ ]:
# ============================================# 1.5.1 生成泊松混合模型data# ============================================# 混合模型parameter（真实值）K_true = 2  # 两个神经元状态pi_true = np.array([0.4, 0.6])  # 混合权重lambda_true_mixture = np.array([2.0, 4.5])  # 两个状态的发火率print("="*60)print("1.5 泊松混合模型")print("="*60)print(f"\n真实混合模型parameter:")print(f"  K = {K_true} 个分量")print(f"  π = {pi_true}  (混合权重)")print(f"  λ = {lambda_true_mixture}  (发火率)")# 生成混合datanp.random.seed(42)T_mixture = 200  # 观察点数z_true = np.random.choice(K_true, size=T_mixture, p=pi_true)  # 隐变量spike_counts_mixture = np.array([np.random.poisson(lambda_true_mixture[z]) for z in z_true])print(f"\n生成data:")print(f"  样本数: {T_mixture}")print(f"  来自分量1 (λ={lambda_true_mixture[0]}) 的样本: {np.sum(z_true==0)}")print(f"  来自分量2 (λ={lambda_true_mixture[1]}) 的样本: {np.sum(z_true==1)}")print(f"\nData summary statistics:")print(f"  均值: {np.mean(spike_counts_mixture):.3f}")print(f"  方差: {np.var(spike_counts_mixture):.3f}")print(f"  预期混合均值: {np.sum(pi_true * lambda_true_mixture):.3f}")

In [ ]:
# ============================================# 1.5.2 可视化混合data# ============================================fig, axes = plt.subplots(2, 2, figsize=(14, 10))# 子图1：观察data直方图ax = axes[0, 0]ax.hist(spike_counts_mixture, bins=np.arange(0, 12) - 0.5, alpha=0.7,        color='steelblue', edgecolor='black', density=True)ax.set_xlabel('尖峰计数', fontsize=11)ax.set_ylabel('频率', fontsize=11)ax.set_title('(a) 观察data的边际分布', fontsize=12, fontweight='bold')ax.grid(True, alpha=0.3)# 子图2：真实分量的理论分布ax = axes[0, 1]k_vals = np.arange(0, 10)for k in range(K_true):    pmf = poisson.pmf(k_vals, lambda_true_mixture[k])    ax.bar(k_vals + k*0.35, pi_true[k] * pmf, width=0.35,           alpha=0.6, label=f'分量{k+1} (π={pi_true[k]:.1f}, λ={lambda_true_mixture[k]:.1f})')ax.set_xlabel('尖峰计数', fontsize=11)ax.set_ylabel('概率', fontsize=11)ax.set_title('(b) 真实混合分布的分量', fontsize=12, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)# 子图3：时间序列（按真实分量着色）ax = axes[1, 0]colors = ['red' if z==0 else 'blue' for z in z_true]ax.scatter(range(T_mixture), spike_counts_mixture, c=colors, alpha=0.6, s=30)ax.axhline(lambda_true_mixture[0], color='red', linestyle='--', alpha=0.5, label='分量1均值')ax.axhline(lambda_true_mixture[1], color='blue', linestyle='--', alpha=0.5, label='分量2均值')ax.set_xlabel('时间 t', fontsize=11)ax.set_ylabel('尖峰计数', fontsize=11)ax.set_title('(c) 时间序列（真实隐变量已知）', fontsize=12, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)# 子图4：箱线图对比两个分量ax = axes[1, 1]data_by_component = [spike_counts_mixture[z_true==k] for k in range(K_true)]bp = ax.boxplot(data_by_component, labels=[f'分量{k+1}' for k in range(K_true)],                patch_artist=True)colors_box = ['lightcoral', 'lightblue']for patch, color in zip(bp['boxes'], colors_box):    patch.set_facecolor(color)ax.set_ylabel('尖峰计数', fontsize=11)ax.set_title('(d) 两个分量的data分布', fontsize=12, fontweight='bold')ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()print("✓ 混合模型data可视化完成")

MAP finds mode of posterior:$$\hat{	heta}_{MAP} = rg\max_	heta P(X|	heta)P(	heta)$$Middle ground between MLE and full Bayesian.

In [ ]:
# ============================================# 1.6.1 实现 EM 算法# ============================================def em_poisson_mixture(X, K, max_iter=100, tol=1e-4, verbose=False):    """    泊松混合模型的 EM 算法    parameter:        X: 观察data (N,)        K: 分量数        max_iter: 最大迭代次数        tol: 收敛容限        verbose: 是否打印迭代信息    返回:        pi: 混合权重 (K,)        lambda_: 发火率 (K,)        gamma: 隐变量后验 (N, K)        log_like: 对数似然轨迹    """    N = len(X)    # 初始化parameter    pi = np.ones(K) / K    lambda_ = np.linspace(np.min(X), np.max(X), K) + np.random.rand(K) * 0.5    lambda_ = np.maximum(lambda_, 0.1)  # 确保正数    gamma = np.zeros((N, K))  # 隐变量后验    log_like_trace = []    if verbose:        print(f"\n{'迭代':<8} {'对数似然':<15} {'变化':<15}")        print("-" * 40)    for iteration in range(max_iter):        # ========== E-Step ==========        # 计算 P(z_t=k | x_t)        for k in range(K):            # P(x_t | z_t=k) * P(z_t=k)            gamma[:, k] = poisson.pmf(X, lambda_[k]) * pi[k]        # 归一化为概率        gamma = gamma / (np.sum(gamma, axis=1, keepdims=True) + 1e-10)        # 计算对数似然        log_like = 0        for i in range(N):            marginal_prob = np.sum([pi[k] * poisson.pmf(X[i], lambda_[k]) for k in range(K)])            log_like += np.log(marginal_prob + 1e-10)        log_like_trace.append(log_like)        # ========== M-Step ==========        # 更新混合权重        N_k = np.sum(gamma, axis=0)  # 每个分量的有效样本数        pi = N_k / N        # 更新发火率        for k in range(K):            lambda_[k] = np.sum(gamma[:, k] * X) / (N_k[k] + 1e-10)        # 检查收敛        if iteration > 0:            change = abs(log_like_trace[-1] - log_like_trace[-2])            if verbose and iteration % 10 == 0:                print(f"{iteration:<8} {log_like_trace[-1]:<15.2f} {change:<15.6f}")            if change < tol:                if verbose:                    print(f"\n收敛于迭代 {iteration}")                break    return pi, lambda_, gamma, np.array(log_like_trace)# 运行 EM 算法print("="*60)print("1.6 期望最大化 (EM) 算法")print("="*60)print(f"\n初始条件:")print(f"  data: {T_mixture} 个观察")print(f"  真实parameter: π={pi_true}, λ={lambda_true_mixture}")print(f"  隐变量: 未知")# 设定要估计的分量数（假设我们知道 K=2）K_est = 2pi_em, lambda_em, gamma_em, log_like_em = em_poisson_mixture(    spike_counts_mixture, K_est, max_iter=100, verbose=True)print(f"\n\n📊 EM 算法估计结果:")print(f"  π_hat = {pi_em}")print(f"  λ_hat = {lambda_em}")print(f"\n📊 真实parameter:")print(f"  π_true = {pi_true}")print(f"  λ_true = {lambda_true_mixture}")print(f"\n📈 parameter估计误差:")# 需要处理分量顺序的可能不同dist1 = np.sum((pi_em - pi_true)**2) + np.sum((lambda_em - lambda_true_mixture)**2)dist2 = np.sum((pi_em[::-1] - pi_true)**2) + np.sum((lambda_em[::-1] - lambda_true_mixture)**2)if dist1 <= dist2:    print(f"  π 误差: {np.linalg.norm(pi_em - pi_true):.4f}")    print(f"  λ 误差: {np.linalg.norm(lambda_em - lambda_true_mixture):.4f}")else:    print(f"  注意：分量顺序可能反转")    print(f"  π 误差 (调整顺序): {np.linalg.norm(pi_em[::-1] - pi_true):.4f}")    print(f"  λ 误差 (调整顺序): {np.linalg.norm(lambda_em[::-1] - lambda_true_mixture):.4f}")

In [ ]:
# ============================================# 1.6.2 可视化 EM 算法收敛# ============================================fig, axes = plt.subplots(2, 2, figsize=(14, 10))# 子图1：对数似然轨迹ax = axes[0, 0]ax.plot(log_like_em, 'b-', linewidth=2.5, marker='o', markersize=4)ax.set_xlabel('EM 迭代次数', fontsize=11)ax.set_ylabel('对数似然', fontsize=11)ax.set_title('(a) EM 算法的对数似然收敛', fontsize=12, fontweight='bold')ax.grid(True, alpha=0.3)# 子图2：隐变量分配ax = axes[0, 1]im = ax.imshow(gamma_em[:100].T, aspect='auto', cmap='RdYlBu_r')ax.set_xlabel('样本索引', fontsize=11)ax.set_ylabel('分量', fontsize=11)ax.set_title('(b) 隐变量软分配 (γ_{t,k}) 前100个样本', fontsize=12, fontweight='bold')ax.set_yticks([0, 1])ax.set_yticklabels(['分量1', '分量2'])plt.colorbar(im, ax=ax, label='概率')# 子图3：估计 vs 真实parameterax = axes[1, 0]x_pos = np.arange(K_est)width = 0.35ax.bar(x_pos - width/2, pi_true, width, label='真实 π', alpha=0.7, color='steelblue')ax.bar(x_pos + width/2, pi_em, width, label='估计 π', alpha=0.7, color='coral')ax.set_xlabel('分量', fontsize=11)ax.set_ylabel('混合权重', fontsize=11)ax.set_title('(c) 混合权重估计对比', fontsize=12, fontweight='bold')ax.set_xticks(x_pos)ax.set_xticklabels([f'k={k+1}' for k in range(K_est)])ax.legend(fontsize=10)ax.grid(True, alpha=0.3, axis='y')# 子图4：发火率估计对比ax = axes[1, 1]ax.bar(x_pos - width/2, lambda_true_mixture, width, label='真实 λ', alpha=0.7, color='steelblue')ax.bar(x_pos + width/2, lambda_em, width, label='估计 λ', alpha=0.7, color='coral')ax.set_xlabel('分量', fontsize=11)ax.set_ylabel('发火率', fontsize=11)ax.set_title('(d) 发火率估计对比', fontsize=12, fontweight='bold')ax.set_xticks(x_pos)ax.set_xticklabels([f'k={k+1}' for k in range(K_est)])ax.legend(fontsize=10)ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()print("\n✓ EM 算法收敛过程可视化完成")

### Summary- MLE: Point estimate from likelihood- MAP: Point estimate with prior regularization- Bayesian: Full posterior distribution- EM: For latent variable models

In [ ]:
# ============================================# 1.7.1 模型选择：AIC 和 BIC# ============================================def compute_model_criteria(X, K, pi, lambda_):    """    计算 AIC 和 BIC    parameter:        X: 观察data        K: 分量数        pi: 混合权重        lambda_: 发火率    返回:        log_like: 对数似然        aic: AIC        bic: BIC    """    N = len(X)    # 计算对数似然    log_like = 0    for i in range(N):        marginal = np.sum([pi[k] * poisson.pmf(X[i], lambda_[k]) for k in range(K)])        log_like += np.log(marginal + 1e-10)    # parameter数量：K 个权重（但有一个约束 Σπ_k=1），K 个发火率    # 实际parameter数：(K-1) + K = 2K - 1    p = 2 * K - 1    # AIC = -2*LL + 2*p    aic = -2 * log_like + 2 * p    # BIC = -2*LL + p*log(N)    bic = -2 * log_like + p * np.log(N)    return log_like, aic, bic# 尝试不同的 K 值K_values = np.arange(1, 6)log_likes = []aics = []bics = []results = []print("="*60)print("1.7 模型选择")print("="*60)print(f"\n尝试不同分量数 K...\n")print(f"{'K':<5} {'对数似然':<15} {'AIC':<15} {'BIC':<15} {'parameter数':<8}")print("-" * 60)for K in K_values:    pi_k, lambda_k, gamma_k, _ = em_poisson_mixture(spike_counts_mixture, K, max_iter=100, verbose=False)    ll, aic, bic = compute_model_criteria(spike_counts_mixture, K, pi_k, lambda_k)    log_likes.append(ll)    aics.append(aic)    bics.append(bic)    results.append((K, pi_k, lambda_k))    p = 2 * K - 1    print(f"{K:<5} {ll:<15.2f} {aic:<15.2f} {bic:<15.2f} {p:<8}")# 找到最优模型best_k_aic = K_values[np.argmin(aics)]best_k_bic = K_values[np.argmin(bics)]print(f"\n📊 模型选择结果:")print(f"  AIC 最优: K = {best_k_aic}")print(f"  BIC 最优: K = {best_k_bic}")print(f"  真实值: K = {K_true}")if best_k_bic == K_true:    print(f"\n  ✓ BIC 正确识别了真实模型!")else:    print(f"\n  注意：BIC 选择的模型与真实值不同")    print(f"  （这可能是由于样本量、parameter值等因素）")

In [ ]:
# ============================================# 1.7.2 可视化模型选择# ============================================fig, axes = plt.subplots(1, 3, figsize=(15, 5))# 子图1：对数似然ax = axes[0]ax.plot(K_values, log_likes, 'o-', linewidth=2.5, markersize=10, color='steelblue')ax.set_xlabel('分量数 K', fontsize=12, fontweight='bold')ax.set_ylabel('对数似然', fontsize=12, fontweight='bold')ax.set_title('(a) 对数似然 vs K', fontsize=12, fontweight='bold')ax.set_xticks(K_values)ax.grid(True, alpha=0.3)# 子图2：AICax = axes[1]ax.plot(K_values, aics, 'o-', linewidth=2.5, markersize=10, color='coral')ax.axvline(best_k_aic, color='red', linestyle='--', linewidth=2, label=f'AIC 最优: K={best_k_aic}')ax.axvline(K_true, color='green', linestyle=':', linewidth=2, label=f'真实: K={K_true}')ax.set_xlabel('分量数 K', fontsize=12, fontweight='bold')ax.set_ylabel('AIC', fontsize=12, fontweight='bold')ax.set_title('(b) AIC vs K', fontsize=12, fontweight='bold')ax.set_xticks(K_values)ax.legend(fontsize=10)ax.grid(True, alpha=0.3)# 子图3：BICax = axes[2]ax.plot(K_values, bics, 'o-', linewidth=2.5, markersize=10, color='lightgreen')ax.axvline(best_k_bic, color='red', linestyle='--', linewidth=2, label=f'BIC 最优: K={best_k_bic}')ax.axvline(K_true, color='green', linestyle=':', linewidth=2, label=f'真实: K={K_true}')ax.set_xlabel('分量数 K', fontsize=12, fontweight='bold')ax.set_ylabel('BIC', fontsize=12, fontweight='bold')ax.set_title('(c) BIC vs K', fontsize=12, fontweight='bold')ax.set_xticks(K_values)ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n✓ 模型选择可视化完成")

### Quiz1. Why mean = variance for Poisson?2. What's the MLE?3. What is conjugate prior?4. Describe EM steps5. When use BIC vs AIC?

### Further ReadingKey papers:- Dempster, Laird, Rubin (1977) - EM- Bishop (2006) - Pattern Recognition and Machine Learning

---**Next**: Apply these concepts to real neural data!